In [1]:
!wget http://submit08.mit.edu/~bmaier/STEAM/data_pileup.npz

--2026-08-18 16:02:14--  http://submit08.mit.edu/~bmaier/STEAM/data_pileup.npz
Resolving submit08.mit.edu (submit08.mit.edu)... 18.4.134.169, 2603:4000:486:1::169
Connecting to submit08.mit.edu (submit08.mit.edu)|18.4.134.169|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1351134430 (1.3G)
Saving to: ‘data_pileup.npz’

data_pileup.npz     100%[===================>]   1.26G  2.17MB/s    in 13m 59s 

2026-08-18 16:16:13 (1.54 MB/s) - ‘data_pileup.npz’ saved [1351134430/1351134430]



# Longformer for pileup mitigation

**Re-enacting PUMA (arXiv:2107.02779) with sliding-window attention.**

Per-particle task: is this particle from a pileup collision? The physics metric is
**MET resolution**, since that is what pileup actually damages.

The experiment:

1. Train a sliding-window transformer with window $w$ to classify each particle.
2. Use the predicted "keep" weight to recompute MET, and compare against `genmet`.
3. Sweep $w$ under **two orderings** of the particles in each event:
   - **native** --- the order already in the file (particles close in detector are grouped together),
   - **pT-sorted** --- descending $p_T$, which destroys that locality.

The claim being tested: with pT sorting you need a wide window to gather related particles,
while the native order hands you locality for free and saturates almost immediately.

Attention memory is tracked throughout.

In [1]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import math, time, gc
import matplotlib.pyplot as plt

torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "| torch", torch.__version__)

FEATS = ['pt', 'eta', 'phi', 'e', 'puppi', 'pdgid', 'cluster_idx', 'vtxid',
                'cluster_r', 'cluster_hardch_pt', 'cluster_puch_pt', 'npv'] #features per particle

device: cuda | torch 2.10.0


---
## 1. Load the file

Point `DATA` at your file. Expected contents:

| array | shape | meaning |
|---|---|---|
| `x` | `(n_events, n_particles, 13)` | the features above, in that order |
| `y` | `(n_events, n_particles)` | 1 = from the hard scatter, 0 = from pileup |
| `genmet`, `genmet_phi` | `(n_events,)` | truth MET |

Events are zero-padded to a common length; a particle is treated as real if $p_T > 0$.

In [14]:
DATA = "./data_pileup.npz"     # <-- 10k simulated events
NPART = 9000                   # particles kept per event (pad/truncate)

def load_real(path):
    d = np.load(path)
    return (d["x"].astype(np.float32), d["y"].astype(np.float32),
            d["met"].astype(np.float32)[:,0], d["met"].astype(np.float32)[:,1])

X, Y, GENMET, GENPHI = load_real(DATA)
print("loaded", DATA)


X, Y = X[:, :NPART], Y[:, :NPART]
MASK = (X[..., 0] > 0).astype(np.float32)          # filtering on pT>0 --> real particle or zero-padded particle?
print("x", X.shape, "| y", Y.shape, "| pileup fraction", round(1-float(Y[MASK==1].mean()),3))
print("mean genMET", round(float(GENMET.mean()),1), "GeV")

ntr = int(0.75*len(X))
sl_tr, sl_te = slice(0,ntr), slice(ntr,len(X))

loaded ./data_pileup.npz
x (10000, 9000, 13) | y (10000, 9000) | pileup fraction 0.974
mean genMET 206.2 GeV


In [15]:
DROP = []
USE  = [i for i, f in enumerate(FEATS) if f not in DROP]
print("Final inputs to be used:", [FEATS[i] for i in USE])

def prep(Xr):
    Z = Xr.copy()
    Z[...,0] = np.log1p(Z[...,0])                                  # pt
    Z[...,3] = np.log1p(Z[...,3])                                  # e
    Z[...,5] = np.sign(Z[...,5])*np.log1p(np.abs(Z[...,5]))        # pdgid
    Z = Z[..., USE]
    flat = Z.reshape(-1, Z.shape[-1])
    mu, sd = flat.mean(0), flat.std(0) + 1e-6
    return ((Z-mu)/sd).astype(np.float32)

def reorder(Xr, Yr, Mr, mode):
    if mode == "native":
        return Xr, Yr, Mr
    key = -Xr[..., 0]                      # descending pt; padding (pt=0) goes last
    o = np.argsort(key, axis=1)
    return (np.take_along_axis(Xr, o[...,None], 1),
            np.take_along_axis(Yr, o, 1),
            np.take_along_axis(Mr, o, 1))

Final inputs to be used: ['pt', 'eta', 'phi', 'e', 'puppi', 'pdgid', 'cluster_idx', 'vtxid', 'cluster_r', 'cluster_hardch_pt', 'cluster_puch_pt', 'npv']


---
## 3. Sliding-window attention, $O(n w)$ 

The $n\times n$ matrix is **never built**. For each query we gather only its $w$ neighbours with
`unfold`, so the score tensor is $(B, H, n, w)$ — linear in $n$.

(If you instead build an $n \times n$ mask and zero the forbidden entries, you get the right
answer and none of the savings. That is the most common way to fake this.)

In [16]:
class BandedAttention(nn.Module):
    def __init__(self, d, nheads, w):
        super().__init__()
        assert w % 2 == 1, "use an odd window so it is symmetric about the query"
        self.h, self.dh, self.w = nheads, d//nheads, w
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.o   = nn.Linear(d, d, bias=False)
        self.score_numel = 0                       # for the memory accounting

    def forward(self, x, key_mask):
        B, N, D = x.shape; H, dh, w = self.h, self.dh, self.w; r = w//2
        q, k, v = self.qkv(x).chunk(3, -1)
        q = q.view(B,N,H,dh).permute(0,2,1,3)
        k = k.view(B,N,H,dh).permute(0,2,1,3)
        v = v.view(B,N,H,dh).permute(0,2,1,3)

        kw = F.pad(k, (0,0,r,r)).unfold(2, w, 1)          # B,H,N,dh,w
        vw = F.pad(v, (0,0,r,r)).unfold(2, w, 1)
        scores = torch.einsum('bhnd,bhndw->bhnw', q, kw) / math.sqrt(dh)
        self.score_numel = scores.numel()

        pos = (torch.arange(N, device=x.device)[:,None]
               + torch.arange(w, device=x.device)[None,:] - r)      # N,w
        inrange = ((pos >= 0) & (pos < N))[None,None]                # 1,1,N,w
        km = F.pad(key_mask, (r,r)).unfold(1, w, 1)                  # B,N,w
        keep = inrange & (km[:,None] > 0)                            # B,1,N,w
        scores = scores.masked_fill(~keep, float('-inf'))
        a = scores.softmax(-1)
        a = torch.nan_to_num(a)                                      # rows with no valid key
        out = torch.einsum('bhnw,bhndw->bhnd', a, vw)
        return self.o(out.permute(0,2,1,3).reshape(B,N,D))

class Block(nn.Module):
    def __init__(self, d, nheads, w, ff):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.att = BandedAttention(d, nheads, w)
        self.ff  = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
    def forward(self, x, m):
        x = x + self.att(self.n1(x), m)
        return x + self.ff(self.n2(x))

class PumaNet(nn.Module):
    def __init__(self, nfeat, w, d=32, nheads=2, nlayers=2, ff=32):
        super().__init__()
        self.inp = nn.Linear(nfeat, d)
        self.blocks = nn.ModuleList([Block(d, nheads, w, ff) for _ in range(nlayers)])
        self.nf, self.head = nn.LayerNorm(d), nn.Linear(d, 1)
    def forward(self, x, m):
        h = self.inp(x)
        for b in self.blocks: h = b(h, m)
        return self.head(self.nf(h)).squeeze(-1)
    def attention_entries(self):
        return sum(b.att.score_numel for b in self.blocks)

m = PumaNet(len(USE), w=33).to(DEV)
print(f"{sum(p.numel() for p in m.parameters()):,} parameters")

13,185 parameters


---
## 4. MET and the memory ledger

Predicted MET uses the model's keep-weight $w_i$ exactly as PUPPI uses its weight:

$$\text{MET}_x = -\sum_i (w_i)\, p_{T,i} \cos\phi_i, \qquad
  \text{MET}_y = -\sum_i (w_i)\, p_{T,i} \sin\phi_i$$

Resolution is the per-component RMS of the residual against truth.

In [20]:
def met_from_weights(Xr, wgt, mask):
    pt, phi = Xr[...,0]*mask, Xr[...,2]
    return -(wgt*pt*np.cos(phi)).sum(1), -(wgt*pt*np.sin(phi)).sum(1)

def met_resolution(mx, my, genmet, genphi):
    """Residuals in each MET component separately.
    Bias is the mean, resolution the width -- keep them apart."""
    gx, gy = genmet*np.cos(genphi), genmet*np.sin(genphi)
    rx, ry = mx - gx, my - gy
    out = {}
    for name, r in (("x", rx), ("y", ry)):
        q16, q84 = np.percentile(r, [16, 84])
        out[f"bias_{name}"] = float(r.mean())          # response offset
        out[f"res_{name}"]  = float(r.std())           # gaussian width
        out[f"q68_{name}"]  = float((q84 - q16)/2)     # robust width
    return out

def attention_bytes(B, H, N, w, nlayers, dh=8, dtype_bytes=4):
    scores = B*H*N*w*nlayers*dtype_bytes
    unfold = B*H*N*dh*w*nlayers*dtype_bytes*2     # kw and vw
    full   = B*H*N*N*nlayers*dtype_bytes
    return scores+unfold, full

def peak_memory_mb():
    if DEV == "cuda":
        return torch.cuda.max_memory_allocated()/1e6
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e3   # KB->MB on Linux

---
## 5. The sweep

For each ordering and each window width: train, evaluate, record MET resolution and memory.

In [ ]:
def run(w, mode, steps=300, bs=8, eval_bs=16, d=16, nlayers=2, nheads=2, lr=3e-3):
    Xo, Yo, Mo = reorder(X, Y, MASK, mode)
    Z = prep(Xo)

    # stay on CPU; only minibatches go to the GPU
    Ztr, Ytr, Mtr = (torch.from_numpy(a[sl_tr]) for a in (Z, Yo, Mo))
    Zte, Mte      = (torch.from_numpy(a[sl_te]) for a in (Z, Mo))

    net = PumaNet(len(USE), w=w, d=d, nheads=nheads, nlayers=nlayers).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.01)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s+1)/40, 1.0))

    if DEV == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for s in range(steps):
        i  = torch.randint(0, len(Ztr), (bs,))
        zb, yb, mb = Ztr[i].to(DEV), Ytr[i].to(DEV), Mtr[i].to(DEV)
        loss = F.binary_cross_entropy_with_logits(net(zb, mb), yb, weight=mb)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sch.step()
    secs = time.time()-t0

    net.eval(); ps = []
    with torch.no_grad():
        for a in range(0, len(Zte), eval_bs):                 # <-- batched eval
            ps.append(torch.sigmoid(
                net(Zte[a:a+eval_bs].to(DEV), Mte[a:a+eval_bs].to(DEV))).cpu().numpy())
    p = np.concatenate(ps)

    msk = Mo[sl_te]
    acc = float((((p>0.5)==(Yo[sl_te]>0.5))*msk).sum()/msk.sum())
    mx, my = met_from_weights(Xo[sl_te], p*msk, msk)          # p, not 1-p
    met = met_resolution(mx, my, GENMET[sl_te], GENPHI[sl_te])
    banded, full = attention_bytes(bs, nheads, NPART, w, nlayers)

    r = dict(mode=mode, w=w, acc=acc, secs=secs, **met,
             attn_MB=banded/1e6, full_MB=full/1e6, peak_MB=peak_memory_mb())
    del net, Ztr, Ytr, Mtr, Zte, Mte, Z
    gc.collect()
    if DEV == "cuda": torch.cuda.empty_cache()
    return r

WINDOWS = [1, 9, 15, 27]#, NPART-1]         # NPART-1 == effectively full attention --> not feasible
results = []
for mode in ["native", "pt"]:
    for w in WINDOWS:
        r = run(w, mode)
        results.append(r)
        print(f"{r['mode']:7s} sorting  w={r['w']:4d}  acc={r['acc']:.5f}  "
              f"res_x={r['res_x']:6.2f}  res_y={r['res_y']:6.2f} GeV   "
              f"bias=({r['bias_x']:+.2f},{r['bias_y']:+.2f})   "
              f"attn={r['attn_MB']:6.2f} MB   {r['secs']:.0f}s", flush=True)